# 01 — Exploração e preparação dos dados

**Trabalho Final — Inteligência de Receita para os Hotéis de Port Alder**

Objetivo deste notebook (Etapa 1 do enunciado):

1. carregar os sete arquivos JSONL do dataset *Nice Rooms*;
2. conferir, para cada tabela, tamanho, granularidade, tipos e valores ausentes;
3. transformar a coluna de comodidades (uma lista dentro de um campo);
4. identificar as colunas que ligam uma tabela à outra.

> O dataset é **sintético**: as conclusões valem para a cidade fictícia de Port Alder, não para o mercado hoteleiro real.

## 1. Preparação do ambiente

As funções `carregar()` (leitura de um JSONL) e `estilo_graficos()` estão no módulo `nice_rooms.py`, nesta mesma pasta, porque os quatro notebooks as usam. As funções de inspeção, que só este notebook usa, são definidas mais abaixo.

In [1]:
import json

import pandas as pd

from nice_rooms import PASTA_DADOS, carregar

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
print("Lendo os dados de:", PASTA_DADOS.resolve())

Lendo os dados de: /Users/walterpaixaocortes/projects/demo-dash/dados


## 2. Como ler um arquivo JSONL

Em um JSONL **cada linha é um objeto JSON independente**. Não é um CSV, então `pd.read_csv` não serve. Também não é um JSON único com uma lista, então `json.load` no arquivo inteiro falha. Há dois caminhos:

- ler linha a linha com `json.loads`, como no Módulo 1 de Python (listas e dicionários);
- usar `pd.read_json(..., lines=True)`, que faz a mesma coisa e já devolve um DataFrame.

Abaixo os dois caminhos são comparados, para mostrar que dão o mesmo resultado.

In [2]:
# Caminho 1: Python puro. Cada linha vira um dicionário.
with open(PASTA_DADOS / "hotels.jsonl", encoding="utf-8") as arquivo:
    hoteis_lista = [json.loads(linha) for linha in arquivo if linha.strip()]

print(type(hoteis_lista), len(hoteis_lista), "registros")
print("Chaves do primeiro registro:", list(hoteis_lista[0].keys()))

# Caminho 2: pandas com lines=True
hoteis_pandas = pd.read_json(PASTA_DADOS / "hotels.jsonl", lines=True)
print("Mesmo número de linhas?", len(hoteis_lista) == len(hoteis_pandas))

<class 'list'> 14 registros
Chaves do primeiro registro: ['hotel_id', 'name', 'brand', 'tier', 'rooms', 'opened_year', 'distance_to_water_km', 'neighbourhood', 'comp_set', 'positioning_index', 'base_rate', 'amenities', 'city', 'review_count', 'lifetime_rating', 'share_rating_5', 'share_rating_1_2']
Mesmo número de linhas? True


Daqui em diante uso a função `carregar(nome)`, que chama `pd.read_json(..., lines=True, convert_dates=False)` e converte **explicitamente** as colunas de data de cada tabela. Desliguei a conversão automática porque o pandas só reconhece datas em algumas colunas (por exemplo, `date` em `daily_market`) e deixa outras como texto (`week_start`, `review_date`). Converter tudo de forma explícita evita essa inconsistência.

In [3]:
NOMES = ["hotels", "room_types", "daily_market", "weekly_performance",
         "rate_decisions", "reviews", "review_scenarios"]

tabelas = {nome: carregar(nome) for nome in NOMES}
for nome, df in tabelas.items():
    print(f"{nome:<20} {df.shape[0]:>6} linhas  {df.shape[1]:>3} colunas")

hotels                   14 linhas   17 colunas
room_types               42 linhas   10 colunas
daily_market          10192 linhas   56 colunas
weekly_performance     1456 linhas   41 colunas
rate_decisions         1442 linhas   24 colunas
reviews               12077 linhas   26 colunas
review_scenarios         11 linhas    7 colunas


## 3. Funções próprias de inspeção

Para não repetir o mesmo código sete vezes, escrevi três funções e as reuso ao longo do notebook:

- `inspecionar(df, nome, linhas_esperadas)`: tamanho, comparação com o README, tipos por categoria e ausentes (incluindo **textos vazios**, que o pandas não conta como ausentes);
- `verificar_chave(df, colunas)`: confirma se uma combinação de colunas identifica cada linha de forma única (granularidade);
- `verificar_ligacao(filha, coluna, mae, coluna_mae, rotulo)`: confirma se todo valor de uma coluna existe na tabela "mãe".

In [4]:
# Números de linhas informados no README do dataset
LINHAS_README = {"hotels": 14, "room_types": 42, "daily_market": 10_192,
                 "weekly_performance": 1_456, "rate_decisions": 1_442,
                 "reviews": 12_077, "review_scenarios": 11}


def classificar_tipo(serie: pd.Series) -> str:
    """Devolve a categoria do tipo de uma coluna: número, texto, data, booleano ou lista."""
    if pd.api.types.is_bool_dtype(serie):
        return "verdadeiro/falso"
    if pd.api.types.is_datetime64_any_dtype(serie):
        return "data"
    if pd.api.types.is_numeric_dtype(serie):
        return "numérica"
    if serie.map(lambda v: isinstance(v, list)).any():
        return "lista"
    return "texto"


def inspecionar(df: pd.DataFrame, nome: str, linhas_esperadas: int) -> pd.DataFrame:
    """Mostra um resumo da tabela e devolve um quadro coluna a coluna."""
    bate = "OK" if len(df) == linhas_esperadas else f"DIFERENTE (README: {linhas_esperadas})"
    print(f"=== {nome}: {df.shape[0]} linhas x {df.shape[1]} colunas | linhas vs README: {bate}")
    resumo = pd.DataFrame({
        "tipo_pandas": df.dtypes.astype(str),
        "categoria": [classificar_tipo(df[c]) for c in df.columns],
        "nulos": df.isna().sum(),
        "texto_vazio": [(df[c] == "").sum() if classificar_tipo(df[c]) == "texto" else 0 for c in df.columns],
        "exemplo": [df[c].iloc[0] for c in df.columns],
    })
    contagem = resumo["categoria"].value_counts().to_dict()
    print("    colunas por categoria:", contagem)
    com_ausentes = resumo[(resumo["nulos"] > 0) | (resumo["texto_vazio"] > 0)]
    if com_ausentes.empty:
        print("    nenhum valor ausente")
    else:
        print("    colunas com ausentes:", ", ".join(
            f"{c} ({r.nulos} nulos, {r.texto_vazio} vazios)" for c, r in com_ausentes.iterrows()))
    return resumo


def verificar_chave(df: pd.DataFrame, colunas: list[str]) -> bool:
    """True se a combinação de colunas não se repete, ou seja, identifica a linha."""
    duplicadas = df.duplicated(subset=colunas).sum()
    print(f"    chave {colunas}: {'única' if duplicadas == 0 else f'{duplicadas} repetições'}")
    return duplicadas == 0


def verificar_ligacao(filha: pd.DataFrame, coluna: str, mae: pd.DataFrame, coluna_mae: str, rotulo: str) -> int:
    """Conta valores da tabela filha que não existem na tabela mãe (órfãos)."""
    orfaos = (~filha[coluna].isin(mae[coluna_mae])).sum()
    print(f"    {rotulo}: {orfaos} órfãos")
    return orfaos

## 4. Inspeção de cada tabela

Para cada tabela: tamanho e comparação com o README, **granularidade** (o que uma linha representa, conferida com `verificar_chave`), tipos e ausentes.

In [5]:
resumos = {}
for nome, df in tabelas.items():
    resumos[nome] = inspecionar(df, nome, LINHAS_README[nome])
    print()

=== hotels: 14 linhas x 17 colunas | linhas vs README: OK
    colunas por categoria: {'numérica': 9, 'texto': 7, 'lista': 1}
    nenhum valor ausente

=== room_types: 42 linhas x 10 colunas | linhas vs README: OK
    colunas por categoria: {'texto': 5, 'numérica': 5}
    nenhum valor ausente

=== daily_market: 10192 linhas x 56 colunas | linhas vs README: OK
    colunas por categoria: {'numérica': 45, 'texto': 10, 'data': 1}
    colunas com ausentes: demand_event (0 nulos, 9534 vazios)

=== weekly_performance: 1456 linhas x 41 colunas | linhas vs README: OK
    colunas por categoria: {'numérica': 35, 'texto': 5, 'data': 1}
    colunas com ausentes: demand_event (0 nulos, 1246 vazios), flow_through (728 nulos, 0 vazios), mean_rating (27 nulos, 0 vazios), positive_share (27 nulos, 0 vazios), sentiment_index (27 nulos, 0 vazios), response_rate (27 nulos, 0 vazios)

=== rate_decisions: 1442 linhas x 24 colunas | linhas vs README: OK
    colunas por categoria: {'numérica': 16, 'texto': 7, '

**Todas as tabelas batem com o README.** Agora a granularidade: a chave candidata de cada tabela precisa ser única.

In [6]:
CHAVES = {
    "hotels": ["hotel_id"],
    "room_types": ["room_type_id"],
    "daily_market": ["hotel_id", "date"],
    "weekly_performance": ["hotel_id", "week_start"],
    "rate_decisions": ["decision_id"],
    "reviews": ["review_id"],
    "review_scenarios": ["scenario_id"],
}
for nome, chave in CHAVES.items():
    print(nome)
    verificar_chave(tabelas[nome], chave)

# A decisão de preço também é única por hotel e semana:
print("rate_decisions (hotel x semana)")
verificar_chave(tabelas["rate_decisions"], ["hotel_id", "week_start"])

hotels
    chave ['hotel_id']: única
room_types
    chave ['room_type_id']: única
daily_market
    chave ['hotel_id', 'date']: única
weekly_performance
    chave ['hotel_id', 'week_start']: única
rate_decisions
    chave ['decision_id']: única
reviews
    chave ['review_id']: única
review_scenarios
    chave ['scenario_id']: única
rate_decisions (hotel x semana)
    chave ['hotel_id', 'week_start']: única


np.True_

| Tabela | Uma linha representa | Chave |
|---|---|---|
| `hotels` | um hotel | `hotel_id` |
| `room_types` | um tipo de quarto de um hotel | `room_type_id` |
| `daily_market` | um hotel em uma noite | `hotel_id` + `date` |
| `weekly_performance` | um hotel em uma semana (segunda a domingo) | `hotel_id` + `week_start` |
| `rate_decisions` | a decisão de preço de um hotel em uma semana, com o resultado da semana seguinte | `decision_id` (também único por `hotel_id` + `week_start`) |
| `reviews` | uma avaliação de hóspede | `review_id` |
| `review_scenarios` | um "motivo" que gera avaliações | `scenario_id` |

Consequência prática: **uma linha diária e uma linha semanal não se somam diretamente**. Para comparar, é preciso agregar a diária por semana, somando quartos e receitas e recalculando os indicadores.

### 4.1 Tipos de dados

Confiro se cada tipo carregado faz sentido. Exemplo para as tabelas com datas e booleanos:

In [7]:
for nome in ["daily_market", "weekly_performance", "rate_decisions", "reviews"]:
    r = resumos[nome]
    print(f"{nome}: datas = {list(r.index[r.categoria == 'data'])}; "
          f"booleanos = {list(r.index[r.categoria == 'verdadeiro/falso'])}")

print()
print(resumos["hotels"][["tipo_pandas", "categoria", "exemplo"]])

daily_market: datas = ['date']; booleanos = []
weekly_performance: datas = ['week_start']; booleanos = []
rate_decisions: datas = ['week_start']; booleanos = []
reviews: datas = ['review_date', 'stay_date', 'week_start']; booleanos = ['verified_stay', 'would_return']

                     tipo_pandas categoria                                            exemplo
hotel_id                     str     texto                                                H01
name                         str     texto                                 Alder & Quay Hotel
brand                        str     texto                                        Independent
tier                         str     texto                                            upscale
rooms                      int64  numérica                                                168
opened_year                int64  numérica                                               1998
distance_to_water_km     float64  numérica                               

Observações sobre tipos:

- As **datas** de todas as tabelas viraram `datetime64` graças à conversão explícita em `carregar()`. Sem ela, `week_start`, `review_date` e `stay_date` ficariam como texto.
- `verified_stay` e `would_return` (reviews) já chegaram como booleanos.
- `rating` e as notas parciais são inteiros, como esperado.
- `amenities` (hotels) chegou como **lista Python**. O README diz que é uma "JSON array" guardada como texto. Como o arquivo grava uma lista JSON de verdade, o `read_json` já a converteu. Trato os dois casos na Seção 6 para o código não quebrar se outra versão do arquivo vier como texto.
- Colunas como `weekday`, `season`, `tier` e `rate_action` são textos com poucos valores possíveis, ou seja, **categóricas**. Converto-as quando a ordem importa (Seção 5).

## 5. Valores ausentes: entender antes de tratar

A inspeção mostrou dois tipos de ausência: **nulos** (`NaN`) e **textos vazios** (`""`). Cada caso tem uma explicação no README, que confiro abaixo com os dados.

In [8]:
semanal = tabelas["weekly_performance"]
diario = tabelas["daily_market"]
avaliacoes = tabelas["reviews"]

# 1) flow_through é nulo no primeiro ano (não há "mesma semana do ano anterior")
nulos_ft = semanal[semanal["flow_through"].isna()]
print("flow_through nulo:", len(nulos_ft), "semanas | anos:", sorted(nulos_ft["week_start"].dt.year.unique()))

# 2) mean_rating etc. são nulos nas semanas sem nenhuma avaliação
sem_nota = semanal[semanal["mean_rating"].isna()]
print("mean_rating nulo:", len(sem_nota), "semanas | review_count nessas semanas:", sem_nota["review_count"].unique())

# 3) demand_event vazio = noite comum (sem evento)
print("noites sem evento:", (diario["demand_event"] == "").sum(), "de", len(diario))

# 4) response_text vazio e response_hours nulo = hotel não respondeu (sempre os dois juntos?)
sem_resposta_texto = avaliacoes["response_text"] == ""
sem_resposta_horas = avaliacoes["response_hours"].isna()
print("avaliações sem resposta:", sem_resposta_texto.sum(),
      "| texto vazio e horas nulas sempre juntos?", (sem_resposta_texto == sem_resposta_horas).all())

# 5) rate_decisions tem 103 semanas por hotel: a última semana não tem semana seguinte para avaliar
print("decisões por hotel:", tabelas["rate_decisions"].groupby("hotel_id").size().unique(),
      "| semanas por hotel:", semanal.groupby("hotel_id").size().unique())

flow_through nulo: 728 semanas | anos: [np.int32(2027)]
mean_rating nulo: 27 semanas | review_count nessas semanas: [0]
noites sem evento: 9534 de 10192
avaliações sem resposta: 3769 | texto vazio e horas nulas sempre juntos? True
decisões por hotel: [103] | semanas por hotel: [104]


**Decisões de tratamento (nenhuma linha é removida):**

| Coluna | Por que está ausente | Decisão |
|---|---|---|
| `flow_through` | 2027 não tem ano anterior para comparar | Manter nulo. Preencher com 0 inventaria um valor. Análises de *flow through* usam só 2028. |
| `mean_rating`, `positive_share`, `sentiment_index`, `response_rate` | 27 semanas sem nenhuma avaliação (`review_count = 0`) | Manter nulo. Uma média de zero avaliações não existe, e preencher com 0 puxaria a nota para baixo. |
| `demand_event` (texto vazio) | Noite ou semana sem evento | Manter o texto e criar a coluna booleana `tem_evento`, mais clara para agrupar. |
| `response_text` vazio / `response_hours` nulo | O hotel não respondeu | Criar a coluna booleana `respondida`. O tempo médio de resposta é calculado só sobre as respondidas. |
| Última semana sem decisão | Não há semana seguinte para medir o resultado | Esperado (103 decisões por hotel, 104 semanas). |

In [9]:
for nome in ["daily_market", "weekly_performance", "rate_decisions"]:
    tabelas[nome]["tem_evento"] = tabelas[nome]["demand_event"] != ""
avaliacoes["respondida"] = avaliacoes["response_text"] != ""

# Categóricas com ordem: tiers, dias da semana e estações
ORDEM_DIAS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
diario["weekday"] = pd.Categorical(diario["weekday"], categories=ORDEM_DIAS, ordered=True)
diario["season"] = pd.Categorical(diario["season"], categories=["winter", "spring", "summer", "autumn"], ordered=True)

print(diario[["date", "weekday", "season", "demand_event", "tem_evento"]].head())
print(avaliacoes["respondida"].value_counts())

        date weekday  season demand_event  tem_evento
0 2027-01-04  Monday  winter                    False
1 2027-01-04  Monday  winter                    False
2 2027-01-04  Monday  winter                    False
3 2027-01-04  Monday  winter                    False
4 2027-01-04  Monday  winter                    False
respondida
True     8308
False    3769
Name: count, dtype: int64


## 6. A coluna com uma lista dentro de um campo: `amenities`

Em `hotels`, cada hotel guarda **várias comodidades em um único campo**. Isso viola a primeira forma normal: não dá para filtrar ou contar comodidades diretamente. A solução é passar do formato **largo** para o **longo** (Aula 03 de Visualização) com `explode`: uma linha por par hotel-comodidade.

In [10]:
hoteis = tabelas["hotels"]


def para_lista(valor) -> list:
    """Garante que o campo seja uma lista, mesmo que venha como texto JSON."""
    return json.loads(valor) if isinstance(valor, str) else list(valor)


hoteis["amenities"] = hoteis["amenities"].map(para_lista)
comodidades = (hoteis[["hotel_id", "name", "amenities"]]
               .explode("amenities")
               .rename(columns={"amenities": "amenity"})
               .reset_index(drop=True))
print(comodidades.shape)
comodidades.head(8)

(57, 3)


,hotel_id,name,amenity
0,H01,Alder & Quay Hotel,harbour view
1,H01,Alder & Quay Hotel,restaurant
2,H01,Alder & Quay Hotel,gym
3,H01,Alder & Quay Hotel,parking
4,H01,Alder & Quay Hotel,pet friendly
5,H02,The Lantern House,design rooms
6,H02,The Lantern House,rooftop bar
7,H02,The Lantern House,breakfast


In [11]:
contagem = comodidades["amenity"].value_counts()
print(contagem.head(10))
print("\nQuantos hotéis têm restaurante (rótulo exato 'restaurant')?",
      comodidades.loc[comodidades["amenity"] == "restaurant", "hotel_id"].nunique())
alguma_refeicao = comodidades[comodidades["amenity"].str.contains("restaurant|dining")]
print("E contando 'two restaurants' e 'fine dining'?", alguma_refeicao["hotel_id"].nunique())
alguma_refeicao

amenity
parking         6
restaurant      4
gym             4
breakfast       4
design rooms    3
spa             3
pool            3
pet friendly    2
laundry         2
free parking    2
Name: count, dtype: int64

Quantos hotéis têm restaurante (rótulo exato 'restaurant')? 4
E contando 'two restaurants' e 'fine dining'? 6


,hotel_id,name,amenity
1,H01,Alder & Quay Hotel,restaurant
11,H03,Marisol Bay Resort,two restaurants
20,H05,The Copper Meridian,fine dining
24,H06,Seabright Grand,restaurant
38,H10,Kestrel Park Hotel,restaurant
55,H14,Willowmere Lodge,restaurant


A pergunta "quantos hotéis têm restaurante?" depende de **como o rótulo é escrito**. Com o rótulo exato `restaurant` são 4 hotéis. Contando `two restaurants` e `fine dining`, a resposta muda. Prefiro a segunda leitura, porque um hotel com "dois restaurantes" tem restaurante, e documento a escolha.

A forma longa também gera uma **tabela de indicadores** (hotel × comodidade, 1 = tem), útil para comparar hotéis:

In [12]:
matriz = pd.crosstab(comodidades["name"], comodidades["amenity"])
matriz[contagem.index[:8]]

amenity,parking,restaurant,gym,breakfast,design rooms,spa,pool,pet friendly
name,,,,,,,,
Alder & Quay Hotel,1,1,1,0,0,0,0,1
Cedar & Salt Hotel,1,0,1,1,0,0,0,0
Harborlight Suites,0,0,1,0,0,0,0,0
Kestrel Park Hotel,1,1,0,0,0,0,1,0
Marisol Bay Resort,0,0,0,0,0,1,1,0
Northgate Inn,0,0,0,0,0,0,0,0
Paloma Court Hotel,1,0,1,1,0,0,0,1
Ridgeline Motor Inn,0,0,0,0,0,0,0,0
Seabright Grand,1,1,0,0,0,0,1,0


## 7. Colunas que ligam as tabelas

As chaves de ligação são `hotel_id` (em todas as tabelas de fatos), `scenario_id` (reviews → review_scenarios), `hotel_id + week_start` (reviews e rate_decisions → weekly_performance) e `hotel_id + room_type` (reviews → room_types). Confiro que nenhum valor fica órfão:

In [13]:
print("Ligações por hotel_id")
for nome in ["room_types", "daily_market", "weekly_performance", "rate_decisions", "reviews"]:
    verificar_ligacao(tabelas[nome], "hotel_id", hoteis, "hotel_id", f"{nome}.hotel_id -> hotels")

print("Ligação por cenário")
verificar_ligacao(avaliacoes, "scenario_id", tabelas["review_scenarios"], "scenario_id",
                  "reviews.scenario_id -> review_scenarios")

print("Ligações compostas (criando uma chave texto hotel|semana)")
semana_chave = semanal["hotel_id"] + "|" + semanal["week_start"].dt.strftime("%Y-%m-%d")
for nome in ["reviews", "rate_decisions"]:
    df = tabelas[nome]
    chave = (df["hotel_id"] + "|" + df["week_start"].dt.strftime("%Y-%m-%d")).to_frame("k")
    verificar_ligacao(chave, "k", semana_chave.to_frame("k"), "k", f"{nome}.(hotel_id, week_start) -> weekly_performance")

quarto_chave = (tabelas["room_types"]["hotel_id"] + "|" + tabelas["room_types"]["room_type"]).to_frame("k")
verificar_ligacao((avaliacoes["hotel_id"] + "|" + avaliacoes["room_type"]).to_frame("k"), "k", quarto_chave, "k",
                  "reviews.(hotel_id, room_type) -> room_types")

Ligações por hotel_id
    room_types.hotel_id -> hotels: 0 órfãos
    daily_market.hotel_id -> hotels: 0 órfãos
    weekly_performance.hotel_id -> hotels: 0 órfãos
    rate_decisions.hotel_id -> hotels: 0 órfãos
    reviews.hotel_id -> hotels: 0 órfãos
Ligação por cenário
    reviews.scenario_id -> review_scenarios: 0 órfãos
Ligações compostas (criando uma chave texto hotel|semana)
    reviews.(hotel_id, week_start) -> weekly_performance: 0 órfãos
    rate_decisions.(hotel_id, week_start) -> weekly_performance: 0 órfãos
    reviews.(hotel_id, room_type) -> room_types: 0 órfãos


np.int64(0)

Nenhum órfão. Essas ligações viram chaves estrangeiras no banco (notebook 03).

## 8. Redundâncias e consistência

Algumas tabelas repetem dados do cadastro (`hotel_name`, `tier`, `comp_set` em `daily_market` e `weekly_performance`; `tier` em `room_types`). Antes de decidir o que fazer com elas no banco, confiro se são **consistentes** com `hotels`. Também confiro algumas identidades do README.

In [14]:
junto = diario.merge(hoteis[["hotel_id", "name", "tier", "comp_set"]], on="hotel_id", suffixes=("", "_cad"))
print("hotel_name igual ao cadastro:", (junto["hotel_name"] == junto["name"]).all())
print("tier igual ao cadastro:", (junto["tier"] == junto["tier_cad"]).all())
print("comp_set igual ao tier:", (hoteis["comp_set"] == hoteis["tier"]).all())
print("available_rooms igual a rooms_available:", (diario["available_rooms"] == diario["rooms_available"]).all())

# Identidades: ocupação = vendidos / disponíveis e RevPAR = receita / disponíveis (tolerância de arredondamento)
erro_occ = (diario["rooms_sold"] / diario["rooms_available"] - diario["occupancy"]).abs().max()
erro_revpar = (diario["room_revenue"] / diario["rooms_available"] - diario["revpar"]).abs().max()
print(f"maior diferença na ocupação: {erro_occ:.5f} | no RevPAR: {erro_revpar:.4f} USD")

# Os resumos de avaliação do cadastro batem com a tabela de avaliações?
calc = avaliacoes.groupby("hotel_id")["rating"].agg(review_count="size", lifetime_rating="mean")
comp = hoteis.set_index("hotel_id")[["review_count", "lifetime_rating"]]
print("review_count bate:", (calc["review_count"] == comp["review_count"]).all(),
      "| lifetime_rating bate:", ((calc["lifetime_rating"] - comp["lifetime_rating"]).abs() < 0.001).all())

# Período coberto
print("período diário:", diario["date"].min().date(), "a", diario["date"].max().date(),
      "| semanas:", semanal["week_start"].nunique())

hotel_name igual ao cadastro: True
tier igual ao cadastro: True
comp_set igual ao tier: True
available_rooms igual a rooms_available: True
maior diferença na ocupação: 0.00005 | no RevPAR: 0.0050 USD
review_count bate: True | lifetime_rating bate: True
período diário: 2027-01-04 a 2028-12-31 | semanas: 104


Tudo consistente. As colunas repetidas são **redundância pura**: cópias exatas que podem ser recuperadas por junção. `available_rooms` é um apelido de `rooms_available`. Os campos `review_count` e `lifetime_rating` do cadastro são **resumos calculáveis** a partir das avaliações. Essas observações alimentam a discussão de normalização do notebook 03.

## 9. Resumo das decisões

1. **Leitura:** `pd.read_json(lines=True)` com conversão explícita das datas.
2. **Linhas:** as sete tabelas batem com o README e as chaves de granularidade são únicas.
3. **Ausentes:** todos são explicados pelo README e **nenhuma linha foi removida nem preenchida**. Criei os indicadores `tem_evento` e `respondida`.
4. **Lista em um campo:** `amenities` virou uma tabela longa `hotel_id × amenity`, que no banco vira a tabela `hotel_amenities`.
5. **Ligações:** `hotel_id`, `scenario_id`, `hotel_id + week_start` e `hotel_id + room_type`, todas sem órfãos.
6. **Cuidados para as próximas etapas:** não somar linhas de granularidades diferentes e não comparar preços entre *tiers* diferentes.